<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

public interface ISearchable
{
    List<Item> SearchByName(string keyword);
}

public interface IReportable
{
    string GenerateReport();
}

public class Item
{
    public string Name { get; private set; }
    public int Volume { get; private set; }
    public string Category { get; private set; }
    public double Weight { get; private set; }
    public double Price { get; private set; }
    public bool IsFragile { get; private set; }
    public DateTime AddedDate { get; private set; }

    public Item(string name, int volume, string category = "Общая", double weight = 0, double price = 0, bool isFragile = false)
    {
        Name = name;
        Volume = volume;
        Category = category;
        Weight = weight;
        Price = price;
        IsFragile = isFragile;
        AddedDate = DateTime.Now;
    }

    public double GetTotalValue(int quantity) => Price * quantity;

    public string CheckCondition() => IsFragile ? "Хрупкий" : "Обычный";

    public override string ToString()
    {
        return $"[Товар: {Name}, Объем: {Volume}, Категория: {Category}, Цена: {Price}]";
    }
}

public abstract class Inventory : ISearchable, IReportable
{
    public int WarehouseId { get; private set; }
    public string WarehouseName { get; private set; }
    public int StorageCapacity { get; private set; }
    public string Location { get; protected set; }
    public bool IsOpen { get; protected set; }
    public string ManagerName { get; protected set; }
    public int MaxItemsCount { get; protected set; }

    protected List<Item> Items { get; private set; }

    public int CurrentVolume => Items.Sum(item => item.Volume);

    protected Inventory(int warehouseId, string warehouseName, int storageCapacity)
    {
        WarehouseId = warehouseId;
        WarehouseName = warehouseName;
        StorageCapacity = storageCapacity;
        Items = new List<Item>();
        Location = "Не указан";
        IsOpen = true;
        ManagerName = "Не назначен";
        MaxItemsCount = 1000;
    }

    public virtual bool CanAddItem(Item item)
    {
        return CurrentVolume + item.Volume <= StorageCapacity && Items.Count < MaxItemsCount;
    }

    public virtual bool AddItem(Item item)
    {
        if (!IsOpen)
        {
            Console.WriteLine($"[Склад '{WarehouseName}'] Ошибка: склад закрыт.");
            return false;
        }
        if (CanAddItem(item))
        {
            Items.Add(item);
            Console.WriteLine($"[Склад '{WarehouseName}'] Товар '{item.Name}' успешно добавлен.");
            return true;
        }
        Console.WriteLine($"[Склад '{WarehouseName}'] Ошибка: Недостаточно места для товара '{item.Name}'.");
        return false;
    }

    public virtual bool RemoveItem(Item item)
    {
        if (Items.Contains(item))
        {
            Items.Remove(item);
            Console.WriteLine($"[Склад '{WarehouseName}'] Товар '{item.Name}' успешно удален.");
            return true;
        }
        Console.WriteLine($"[Склад '{WarehouseName}'] Ошибка: Товар '{item.Name}' не найден на складе.");
        return false;
    }

    public virtual bool TransferItem(Item item, Inventory targetWarehouse)
    {
        if (this.Items.Contains(item) && targetWarehouse.CanAddItem(item))
        {
            this.RemoveItem(item);
            targetWarehouse.AddItem(item);
            Console.WriteLine($"[Трансфер] Товар '{item.Name}' успешно перемещен со склада '{this.WarehouseName}' на '{targetWarehouse.WarehouseName}'.");
            return true;
        }
        Console.WriteLine($"[Трансфер] Не удалось переместить '{item.Name}'. Проверьте наличие товара и вместимость склада '{targetWarehouse.WarehouseName}'.");
        return false;
    }

    public List<Item> GetItemsByCategory(string category)
    {
        return Items.Where(i => i.Category.Equals(category, StringComparison.OrdinalIgnoreCase)).ToList();
    }

    public List<Item> SearchByName(string keyword)
    {
        return Items.Where(i => i.Name.Contains(keyword, StringComparison.OrdinalIgnoreCase)).ToList();
    }

    public virtual string GenerateReport()
    {
        return $"Склад '{WarehouseName}' (ID: {WarehouseId}): позиций {Items.Count}, объем {CurrentVolume}/{StorageCapacity}, открыт: {IsOpen}";
    }

    public virtual void GetStorageStatus()
    {
        int freeSpace = StorageCapacity - CurrentVolume;
        Console.WriteLine($"--- Статус склада '{WarehouseName}' (ID: {WarehouseId}) ---");
        Console.WriteLine($"Вместимость: {StorageCapacity} | Занято: {CurrentVolume} | Свободно: {freeSpace}");
        Console.WriteLine($"Локация: {Location} | Открыт: {IsOpen} | Управляющий: {ManagerName}");
    }

    public void Close() { IsOpen = false; Console.WriteLine($"Склад '{WarehouseName}' закрыт."); }
    public void Open() { IsOpen = true; Console.WriteLine($"Склад '{WarehouseName}' открыт."); }
}

public class PersonalInventory : Inventory
{
    public string OwnerName { get; private set; }
    public string SecurityCode { get; private set; }
    public bool IsLocked { get; private set; }
    public int MaxItemWeight { get; private set; }

    public PersonalInventory(int warehouseId, string warehouseName, int storageCapacity, string ownerName)
        : base(warehouseId, warehouseName, storageCapacity)
    {
        OwnerName = ownerName;
        SecurityCode = "0000";
        IsLocked = false;
        MaxItemWeight = 50;
        Location = "Дом";
        ManagerName = ownerName;
    }

    public bool Lock(string code)
    {
        if (code == SecurityCode) { IsLocked = true; Console.WriteLine($"Склад '{WarehouseName}' заблокирован."); return true; }
        Console.WriteLine("Неверный код!");
        return false;
    }

    public bool Unlock(string code)
    {
        if (code == SecurityCode) { IsLocked = false; Console.WriteLine($"Склад '{WarehouseName}' разблокирован."); return true; }
        Console.WriteLine("Неверный код!");
        return false;
    }

    public override bool AddItem(Item item)
    {
        if (IsLocked) { Console.WriteLine($"[Личный склад] Ошибка: склад заблокирован."); return false; }
        if (item.Weight > MaxItemWeight) { Console.WriteLine($"[Личный склад] Ошибка: вес товара {item.Weight} кг превышает лимит {MaxItemWeight} кг."); return false; }
        return base.AddItem(item);
    }

    public override void GetStorageStatus()
    {
        base.GetStorageStatus();
        Console.WriteLine($"Владелец: {OwnerName} | Заблокирован: {IsLocked}");
        Console.WriteLine("---------------------------------------------");
    }

    public override string GenerateReport()
    {
        return base.GenerateReport() + $", владелец: {OwnerName}, заблокирован: {IsLocked}";
    }
}

public class GroupInventory : Inventory
{
    public string ProductGroup { get; private set; }
    public string Department { get; private set; }
    public double Budget { get; private set; }
    public int EmployeeCount { get; private set; }

    public GroupInventory(int warehouseId, string warehouseName, int storageCapacity, string productGroup)
        : base(warehouseId, warehouseName, storageCapacity)
    {
        ProductGroup = productGroup;
        Department = "Общий";
        Budget = 0;
        EmployeeCount = 0;
        Location = "Офис";
    }

    public void SetBudget(double budget) { Budget = budget; Console.WriteLine($"[Групповой склад] Бюджет установлен: {budget} руб."); }
    public void AddEmployee() { EmployeeCount++; Console.WriteLine($"[Групповой склад] Сотрудник добавлен. Всего: {EmployeeCount}"); }

    public override bool AddItem(Item item)
    {
        if (!item.Category.Equals(ProductGroup, StringComparison.OrdinalIgnoreCase))
        {
            Console.WriteLine($"[Групповой склад] Ошибка: Товар '{item.Name}' (Категория: {item.Category}) не соответствует группе склада '{ProductGroup}'.");
            return false;
        }
        if (base.AddItem(item))
        {
            Console.WriteLine($"[Групповой склад] Товар '{item.Name}' успешно отнесен к товарной группе '{ProductGroup}'.");
            return true;
        }
        return false;
    }

    public override void GetStorageStatus()
    {
        base.GetStorageStatus();
        Console.WriteLine($"Группа товаров: {ProductGroup} | Отдел: {Department} | Бюджет: {Budget} руб. | Сотрудников: {EmployeeCount}");
        Console.WriteLine("---------------------------------------------");
    }

    public override string GenerateReport()
    {
        return base.GenerateReport() + $", группа: {ProductGroup}, отдел: {Department}, бюджет: {Budget}";
    }
}

public class AutomatedInventory : Inventory
{
    public string AutomationLevel { get; private set; }
    public string RobotModel { get; private set; }
    public string SoftwareVersion { get; private set; }
    public double Efficiency { get; private set; }

    public AutomatedInventory(int warehouseId, string warehouseName, int storageCapacity, string automationLevel)
        : base(warehouseId, warehouseName, storageCapacity)
    {
        AutomationLevel = automationLevel;
        RobotModel = "Стандартный";
        SoftwareVersion = "1.0.0";
        Efficiency = 90.0;
        Location = "Автоматизированный цех";
    }

    public void UpdateSoftware(string version)
    {
        SoftwareVersion = version;
        Console.WriteLine($"[Автоматизированный склад] ПО обновлено до версии {version}.");
    }

    public override bool RemoveItem(Item item)
    {
        if (base.RemoveItem(item))
        {
            Console.WriteLine($"[Автоматизированный склад] Уровень автоматизации: {AutomationLevel}. Робот-погрузчик завершил операцию удаления товара '{item.Name}'.");
            return true;
        }
        return false;
    }

    public override void GetStorageStatus()
    {
        base.GetStorageStatus();
        Console.WriteLine($"Автоматизация: {AutomationLevel} | Робот: {RobotModel} | ПО: {SoftwareVersion} | Эффективность: {Efficiency}%");
        Console.WriteLine("---------------------------------------------");
    }

    public override string GenerateReport()
    {
        return base.GenerateReport() + $", автоматизация: {AutomationLevel}, робот: {RobotModel}, ПО: {SoftwareVersion}";
    }
}

public class SmartAutomatedInventory : AutomatedInventory
{
    public bool HasClimateControl { get; private set; }
    public double Temperature { get; private set; }
    public double Humidity { get; private set; }
    public DateTime LastInspectionDate { get; private set; }

    public SmartAutomatedInventory(int warehouseId, string warehouseName, int storageCapacity, string automationLevel)
        : base(warehouseId, warehouseName, storageCapacity, automationLevel)
    {
        HasClimateControl = true;
        Temperature = 20.0;
        Humidity = 45.0;
        LastInspectionDate = DateTime.Now;
    }

    public void CheckClimate()
    {
        Console.WriteLine($"[Умный склад] Температура: {Temperature}°C, Влажность: {Humidity}%, Климат-контроль: {(HasClimateControl ? "активен" : "отключен")}");
    }

    public void SetTemperature(double temp)
    {
        Temperature = temp;
        Console.WriteLine($"[Умный склад] Температура установлена: {temp}°C");
    }

    public override void GetStorageStatus()
    {
        base.GetStorageStatus();
        Console.WriteLine($"Климат-контроль: {(HasClimateControl ? "да" : "нет")} | Температура: {Temperature}°C | Влажность: {Humidity}%");
        Console.WriteLine("---------------------------------------------");
    }

    public override string GenerateReport()
    {
        return base.GenerateReport() + $", климат-контроль: {(HasClimateControl ? "да" : "нет")}, температура: {Temperature}°C";
    }
}